In [1]:
import pandas as pd
import re

In [2]:
df=pd.read_csv(r'data\dataset.csv')
df.head()

,Role,Resume,Decision,Reason_for_decision,Job_Description
0,E-commerce Specialist,Here's a professional resume for Jason Jones:\...,reject,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
1,Game Developer,Here's a professional resume for Ann Marshall:...,select,Strong technical skills in AI and ML.,Help us build the next-generation products as ...
2,Human Resources Specialist,Here's a professional resume for Patrick Mccla...,reject,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...
3,E-commerce Specialist,Here's a professional resume for Patricia Gray...,select,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
4,E-commerce Specialist,Here's a professional resume for Amanda Gross:...,reject,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...


In [3]:
# 2. Inspect the columns (The dataset usually has columns like 'Resume_str', 'Category', 'Decision', 'Reason', etc.)
print("Original Columns:", df.columns.tolist())

# Let's standardize the column names to match the pipeline we built
# Note: You might need to adjust these mappings slightly depending on the exact 
# capitalization in the dataset, but this is the standard structure.
df = df.rename(columns={
    'Role': 'role',
    'Resume': 'resume', 
    'Job_Description': 'job_description', # Or whatever the JD column is named
    'Decision': 'result_text',
    'Reason_for_decision': 'reason'
})
df.columns.tolist()

Original Columns: ['Role', 'Resume', 'Decision', 'Reason_for_decision', 'Job_Description']


['role', 'resume', 'result_text', 'reason', 'job_description']

In [4]:
# 3. Clean and format the Target Label (Result)
# The dataset likely has "select" or "reject" as text. The Random Forest needs 1 or 0.
def encode_decision(text):
    if not isinstance(text, str):
        return 0
    return 1 if 'select' in text.lower() else 0

df['result'] = df['result_text'].apply(encode_decision)

In [5]:
# 4. Text Cleaning Function (Lightweight cleaning so we don't destroy Transformer context)
def clean_for_llm(text):
    if not isinstance(text, str):
        return ""
    # Remove extra whitespace and newlines but keep punctuation for the Transformer
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['role'] = df['role'].apply(clean_for_llm)
df['job_description'] = df['job_description'].apply(clean_for_llm)
df['resume'] = df['resume'].apply(clean_for_llm)
df['reason'] = df['reason'].apply(clean_for_llm)

In [6]:
# 5. Create the Combined Input for Task 1 (Classification) and Task 2 (Generation)
df['input_text'] = "role: " + df['role'] + " jd: " + df['job_description'] + " resume: " + df['resume']

# 6. Save the prepared data for Notebooks 2, 3, and 4
final_df = df[['input_text', 'result', 'reason']]
final_df.to_csv('data/processed_resume_match.csv', index=False)

print(f"Dataset processing complete. Prepared {len(final_df)} resumes.")
print(final_df.head())

Dataset processing complete. Prepared 10174 resumes.
                                          input_text  result  \
0  role: E-commerce Specialist jd: Be part of a p...       0   
1  role: Game Developer jd: Help us build the nex...       1   
2  role: Human Resources Specialist jd: We need a...       0   
3  role: E-commerce Specialist jd: Be part of a p...       1   
4  role: E-commerce Specialist jd: We are looking...       0   

                                              reason  
0    Lacked leadership skills for a senior position.  
1              Strong technical skills in AI and ML.  
2  Insufficient system design expertise for senio...  
3  Impressive leadership and communication abilit...  
4    Lacked leadership skills for a senior position.  
